# Desafio: RNN Vanilla vs. LSTM vs. GRU em Dados Reais

Implemente e compare as três arquiteturas recorrentes (**RNN Vanilla**, **LSTM** e **GRU**) em três problemas
reais de séries temporais:

1. **Mercado financeiro** — previsão do preço de uma ação (Mastercard, ticker `MA`).
2. **Saúde pública** — previsão de casos diários de COVID-19.
3. **Clima** — previsão de temperatura (estação meteorológica de Jena, Alemanha).

**Como funciona este caderno**: as células marcadas com `# TODO` estão incompletas — complete-as antes de
rodar. As perguntas de reflexão têm um espaço reservado para sua resposta logo abaixo do enunciado. Um
notebook separado (`..._gabarito.ipynb`) traz a resolução completa — só consulte depois de tentar.

**Fontes de dados (treinamento)**:

| Estudo de caso | Fonte | Link |
| --- | --- | --- |
| Ações (Mastercard, `MA`) | Yahoo Finance, via biblioteca `yfinance` | https://finance.yahoo.com/quote/MA/history |
| COVID-19 (casos confirmados) | CSSE / Johns Hopkins University (GitHub) | https://github.com/CSSEGISandData/COVID-19 |
| Temperatura (Jena, Alemanha) | Max Planck Institute for Biogeochemistry | https://www.bgc-jena.mpg.de/wetter/ |

Os links de *download direto* (usados no código) estão na introdução de cada estudo de caso, mais abaixo.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)


## 1. Funções e Arquiteturas Comuns

Estas funções e classes são usadas nos **três** estudos de caso — defina-as uma vez aqui.

### 1.1 Janelamento

Transforma uma série 1D em pares `(janela de entrada, próximo valor)`. Já implementada — reaproveite nos
três estudos de caso.


In [ ]:
def create_dataset(data, time_steps=10):
    '''Transforma uma série 1D em janelas deslizantes.

    Retorna X com shape (n_amostras, time_steps) e y com shape (n_amostras,).
    '''
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i + time_steps])
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)


def to_tensors(X, y):
    '''Converte arrays 2D (n_amostras, time_steps) para tensores 3D (batch, seq_len, n_features=1).'''
    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)
    return X_t, y_t


### 1.2 As três arquiteturas

**TODO**: complete as três classes abaixo. Cada uma segue o mesmo padrão:
`camada recorrente (batch_first=True) -> pega o último passo de tempo -> camada Linear`.
Use `nn.RNN`, `nn.LSTM` e `nn.GRU`, respectivamente — a interface das três é idêntica.


In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()
        # TODO: crie a camada recorrente nn.RNN(...) com batch_first=True
        self.rnn = None  # <-- complete aqui
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # TODO: passe x pela camada recorrente e use apenas o último passo de tempo
        raise NotImplementedError


class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()
        # TODO: crie a camada recorrente nn.LSTM(...) com batch_first=True
        self.lstm = None  # <-- complete aqui
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # TODO: passe x pela camada recorrente e use apenas o último passo de tempo
        raise NotImplementedError


class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, output_size=1):
        super().__init__()
        # TODO: crie a camada recorrente nn.GRU(...) com batch_first=True
        self.gru = None  # <-- complete aqui
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # TODO: passe x pela camada recorrente e use apenas o último passo de tempo
        raise NotImplementedError


### 1.3 Treinamento e avaliação

Funções genéricas de treino e avaliação — já implementadas, reaproveite nos três estudos de caso.


In [ ]:
def train_model(model, loader, epochs=30, lr=1e-3, verbose_every=10):
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for seq, labels in loader:
            optimizer.zero_grad()
            y_pred = model(seq)
            loss = loss_fn(y_pred, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        if (epoch + 1) % verbose_every == 0:
            print(f"  Época {epoch + 1:3d} | Loss treino: {avg_loss:.5f}")
    return losses


def evaluate_model(model, X_test_t, y_test_t, scaler=None):
    '''Avalia o modelo no teste. Se um scaler for passado, desfaz a normalização antes de calcular o erro.'''
    model.eval()
    with torch.no_grad():
        pred = model(X_test_t).numpy()
    true = y_test_t.numpy()
    if scaler is not None:
        pred = scaler.inverse_transform(pred)
        true = scaler.inverse_transform(true)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    mae = float(mean_absolute_error(true, pred))
    return {"rmse": rmse, "mae": mae, "pred": pred, "true": true}


def run_three_architectures(train_loader, X_test_t, y_test_t, hidden_size=32, epochs=30, lr=1e-3, scaler=None):
    '''Treina VanillaRNN, LSTMModel e GRUModel nos mesmos dados e retorna um dicionário de resultados.'''
    results = {}
    for name, ModelClass in [("RNN Vanilla", VanillaRNN), ("LSTM", LSTMModel), ("GRU", GRUModel)]:
        print(f"Treinando {name}...")
        model = ModelClass(hidden_size=hidden_size)
        losses = train_model(model, train_loader, epochs=epochs, lr=lr)
        metrics = evaluate_model(model, X_test_t, y_test_t, scaler=scaler)
        results[name] = {"model": model, "losses": losses, **metrics}
        print(f"  {name}: RMSE={metrics['rmse']:.4f} | MAE={metrics['mae']:.4f}")
    return results


def plot_comparison(results, title, true_label="Real"):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for name, r in results.items():
        axes[0].plot(r["losses"], label=name)
    axes[0].set_title(f"Loss de treino — {title}")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("MSE")
    axes[0].legend()

    any_result = next(iter(results.values()))
    axes[1].plot(any_result["true"], label=true_label, color="grey", alpha=0.6)
    for name, r in results.items():
        axes[1].plot(r["pred"], label=f"Predito ({name})", linewidth=1.5, alpha=0.85)
    axes[1].set_title(f"Predição vs. Real — {title}")
    axes[1].set_xlabel("Passo de tempo (teste)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def summary_table(results):
    rows = [{"Modelo": name, "RMSE": r["rmse"], "MAE": r["mae"]} for name, r in results.items()]
    return pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)


## 2. Estudo de Caso 1 — Mercado Financeiro (ação Mastercard)

**Problema de negócio**: prever o preço mais alto do dia (`High`) da ação da Mastercard (`MA`) a partir do
histórico recente, e decidir qual arquitetura recorrente generaliza melhor para dados financeiros
(ruidosos, não estacionários).

**Fonte dos dados**: Yahoo Finance, via biblioteca `yfinance` (não requer conta/API key).
Link para consulta manual: https://finance.yahoo.com/quote/MA/history
Alternativa (CSV estático): dataset "Stock Prices Of MasterCard And Visa (2008–2024)" no Kaggle —
https://www.kaggle.com/datasets/ranatalha71/stock-prices-of-mastercard-and-visa-2008-2024

> Requer `pip install yfinance` e conexão com a internet.


In [ ]:
import yfinance as yf

stock_df = yf.download("MA", period="5y", interval="1d", progress=False)
stock_df = stock_df[["High"]].dropna()
print(stock_df.shape)
stock_df.tail()


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(stock_df.index, stock_df["High"])
plt.title("Mastercard (MA) — preço 'High' diário")
plt.xlabel("Data")
plt.ylabel("Preço (US$)")
plt.show()


**TODO**: complete o pré-processamento.
1. Separe 80% para treino e 20% para teste (**sem embaralhar** — é série temporal).
2. Ajuste um `MinMaxScaler` **apenas nos dados de treino** e aplique-o em treino e teste.
3. Use `create_dataset` com uma janela de 20 dias para gerar `X`/`y` de treino e teste.
4. Converta para tensores com `to_tensors` e monte um `DataLoader` (`batch_size=16`, `shuffle=False`).


In [ ]:
STOCK_WINDOW = 20

high_values = stock_df["High"].values.reshape(-1, 1)

# TODO: defina stock_train_size (80% dos dados) e separe stock_train / stock_test
stock_train_size = None
stock_train, stock_test = None, None

# TODO: ajuste o MinMaxScaler apenas no treino e transforme treino e teste
stock_scaler = MinMaxScaler()
stock_train_scaled = None
stock_test_scaled = None

# TODO: use create_dataset(..., STOCK_WINDOW) nos dados escalados (achatados com .flatten())
Xs_train, ys_train = None, None
Xs_test, ys_test = None, None

# TODO: converta para tensores com to_tensors e monte o DataLoader (batch_size=16, shuffle=False)
Xs_train_t, ys_train_t = None, None
Xs_test_t, ys_test_t = None, None
stock_loader = None

print("Xs_train_t:", Xs_train_t.shape if Xs_train_t is not None else None)


**TODO**: use `run_three_architectures` para treinar e comparar RNN Vanilla, LSTM e GRU neste dataset
(lembre-se de passar `scaler=stock_scaler`, já que os dados estão normalizados).


In [ ]:
# TODO: chame run_three_architectures(...) com os dados de ações
stock_results = None


In [ ]:
plot_comparison(stock_results, title="Mastercard (MA) — preço High")
summary_table(stock_results)


**Pergunta 1.1** — Qual arquitetura obteve o menor RMSE neste dataset? Isso já era esperado, dado o tamanho e o ruído da série? Justifique.

---

**Sua resposta:**

_(escreva aqui)_

**Pergunta 1.2** — Por que foi necessário desfazer a normalização (`inverse_transform`) antes de calcular o RMSE, em vez de calculá-lo direto nos valores escalados (entre 0 e 1)?

---

**Sua resposta:**

_(escreva aqui)_

## 3. Estudo de Caso 2 — Casos Diários de COVID-19

**Problema de negócio**: prever o número de novos casos confirmados de COVID-19 por dia para um país, e
gerar uma previsão autoregressiva de múltiplos dias à frente (útil para planejamento de capacidade
hospitalar).

**Fonte dos dados**: repositório oficial do CSSE da Johns Hopkins University, no GitHub.
Link do repositório: https://github.com/CSSEGISandData/COVID-19
Link de download direto (CSV): https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv


In [ ]:
COVID_URL = (
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
    "csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
)

covid_raw = pd.read_csv(COVID_URL)
print(covid_raw.shape)
covid_raw.head(3)


**TODO**: complete o pré-processamento.
1. Filtre a linha (ou linhas) do país `"Brazil"` em `covid_raw` e some as colunas de data (agregando
   províncias, se houver) para obter a série **acumulada** de casos.
2. Converta o índice para `datetime` (formato `"%m/%d/%y"`).
3. Transforme a série acumulada em **casos novos por dia** com `.diff()`, tratando valores ausentes/negativos
   (`.fillna(0).clip(lower=0)`).


In [ ]:
country = "Brazil"

# TODO: filtre covid_raw para o país acima e some as colunas de data (covid_raw.columns[4:])
cumulative = None

# TODO: converta cumulative.index para datetime (format="%m/%d/%y")

# TODO: calcule os casos novos por dia (daily_new) a partir de 'cumulative'
daily_new = None

plt.figure(figsize=(14, 4))
plt.plot(daily_new)
plt.title(f"Casos novos diários de COVID-19 — {country}")
plt.show()


**TODO**: repita o fluxo de normalização + janelamento + `DataLoader` do Estudo de Caso 1, agora para
`daily_new` (janela de 5 dias, `batch_size=8`).


In [ ]:
COVID_WINDOW = 5
covid_values = daily_new.values.astype(float).reshape(-1, 1)

# TODO: split 80/20, scaler ajustado só no treino, create_dataset, to_tensors, DataLoader (batch_size=8)
covid_train_size = None
covid_train, covid_test = None, None
covid_scaler = MinMaxScaler()
covid_train_scaled = None
covid_test_scaled = None
Xc_train, yc_train = None, None
Xc_test, yc_test = None, None
Xc_train_t, yc_train_t = None, None
Xc_test_t, yc_test_t = None, None
covid_loader = None


In [ ]:
# TODO: chame run_three_architectures(...) com os dados de COVID-19
covid_results = None


In [ ]:
plot_comparison(covid_results, title=f"Casos novos diários — {country}")
summary_table(covid_results)


### Previsão autoregressiva multi-passo

**TODO**: usando o **melhor modelo** (menor RMSE na tabela acima), implemente uma previsão autoregressiva
de `DAYS_TO_PREDICT=12` dias: a cada passo, use a própria predição do modelo como entrada para prever o
passo seguinte (deslizando a janela).


In [ ]:
DAYS_TO_PREDICT = 12

# TODO: escolha o melhor modelo pelo RMSE em covid_results
best_name = None
best_model = None
best_model.eval()

last_window = torch.tensor(
    covid_train_scaled[-COVID_WINDOW:].flatten(), dtype=torch.float32
).view(1, COVID_WINDOW, 1)

future_preds_scaled = []
current_seq = last_window.clone()
with torch.no_grad():
    for _ in range(DAYS_TO_PREDICT):
        # TODO: preveja o próximo valor com best_model, salve em future_preds_scaled,
        # e desloque a janela (remova o mais antigo, adicione a predição no final)
        pass

future_preds = covid_scaler.inverse_transform(np.array(future_preds_scaled).reshape(-1, 1))

plt.figure(figsize=(14, 5))
plt.plot(daily_new.values[-60:], label="Observado (últimos 60 dias)")
plt.plot(
    range(59, 59 + DAYS_TO_PREDICT + 1),
    [daily_new.values[-1]] + list(future_preds.flatten()),
    label=f"Previsão ({best_name}, +{DAYS_TO_PREDICT} dias)",
    linestyle="--",
)
plt.legend()
plt.title(f"Previsão autoregressiva — {country}")
plt.show()


**Pergunta 2.1** — Por que a série de casos **acumulados** não é usada diretamente (sem `.diff()`) para treinar o modelo?

---

**Sua resposta:**

_(escreva aqui)_

**Pergunta 2.2** — Por que o erro da previsão autoregressiva tende a crescer conforme os dias avançam?

---

**Sua resposta:**

_(escreva aqui)_

## 4. Estudo de Caso 3 — Temperatura (Estação Meteorológica de Jena)

**Problema de negócio**: prever a temperatura da próxima hora a partir do histórico recente — o mesmo tipo
de problema de forecasting climático usado em previsão de demanda de energia (aquecimento/refrigeração).

**Fonte dos dados**: Max Planck Institute for Biogeochemistry, estação de Jena, Alemanha (dataset clássico
usado no tutorial oficial de séries temporais do TensorFlow/Keras).
Página oficial: https://www.bgc-jena.mpg.de/wetter/
Tutorial de referência (Keras): https://keras.io/examples/timeseries/timeseries_weather_forecasting/

Link de download direto (CSV, mirror no GitHub — use este se o link oficial do Google Storage
`https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip` estiver
bloqueado na sua rede):
https://raw.githubusercontent.com/gilbutITbook/006975/master/datasets/jena_climate/jena_climate_2009_2016.csv

O arquivo original tem uma medição a cada 10 minutos entre 2009 e 2016 (~420 mil linhas). Para manter o
treinamento rápido, vamos reamostrar para **1 medição por hora** (prática comum para este dataset).


In [ ]:
JENA_URL_OFICIAL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
JENA_URL_MIRROR = "https://raw.githubusercontent.com/gilbutITbook/006975/master/datasets/jena_climate/jena_climate_2009_2016.csv"

try:
    jena_raw = pd.read_csv(JENA_URL_OFICIAL)
except Exception as e:
    print(f"Link oficial indisponível ({e}); usando o mirror do GitHub.")
    jena_raw = pd.read_csv(JENA_URL_MIRROR)

print(jena_raw.shape)
jena_raw.head(3)


**TODO**: complete o pré-processamento.
1. Selecione a coluna de temperatura `"T (degC)"`.
2. Reamostre para 1 medição por hora, pegando **a cada 6ª linha** (o dataset original é a cada 10 min →
   `jena_raw.iloc[::6]`).
3. Separe 80/20 treino/teste, ajuste o `MinMaxScaler` no treino, janele (janela de **24 horas**) e monte o
   `DataLoader` (`batch_size=32`).


In [ ]:
JENA_WINDOW = 24

# TODO: selecione a coluna "T (degC)" e reamostre com .iloc[::6]
temperature = None

plt.figure(figsize=(14, 4))
plt.plot(temperature.values[:24 * 90])  # ~90 dias, só para visualizar
plt.title("Temperatura horária em Jena (amostra: ~90 dias)")
plt.xlabel("Horas")
plt.ylabel("°C")
plt.show()

temp_values = temperature.values.reshape(-1, 1)

# TODO: split 80/20, scaler ajustado só no treino, create_dataset, to_tensors, DataLoader (batch_size=32)
jena_train_size = None
jena_train, jena_test = None, None
jena_scaler = MinMaxScaler()
jena_train_scaled = None
jena_test_scaled = None
Xj_train, yj_train = None, None
Xj_test, yj_test = None, None
Xj_train_t, yj_train_t = None, None
Xj_test_t, yj_test_t = None, None
jena_loader = None


**Nota de desempenho**: o dataset completo de treino tem dezenas de milhares de janelas. Para manter o
treino rápido em CPU, vamos usar um subconjunto (`JENA_TRAIN_LIMIT` janelas) — em um ambiente com GPU,
sinta-se à vontade para remover esse limite.


In [ ]:
JENA_TRAIN_LIMIT = 4000  # nº de janelas de treino usadas (subamostragem p/ treino rápido em CPU)

# TODO: crie jena_loader_sub usando só as primeiras JENA_TRAIN_LIMIT janelas de Xj_train_t/yj_train_t
jena_loader_sub = None

# TODO: chame run_three_architectures(...) com jena_loader_sub, Xj_test_t, yj_test_t, scaler=jena_scaler
jena_results = None


In [ ]:
plot_comparison(jena_results, title="Temperatura horária — Jena")
summary_table(jena_results)


**Pergunta 3.1** — A temperatura tem um padrão diário bem definido (mais quente de dia, mais fria à noite). Dada a janela de 24 horas usada, você esperaria que o modelo capture esse padrão com facilidade? Por quê?

---

**Sua resposta:**

_(escreva aqui)_

**Pergunta 3.2** — Diferente dos dois estudos de caso anteriores, aqui usamos um `JENA_TRAIN_LIMIT` para reduzir o número de janelas de treino. Que efeito isso provavelmente tem na comparação entre as três arquiteturas, e como você investigaria se a conclusão muda com o dataset completo?

---

**Sua resposta:**

_(escreva aqui)_

## 5. Comparação Final entre os Três Estudos de Caso

**TODO**: monte uma tabela única juntando os resultados dos três estudos de caso (uma linha por
combinação dataset × arquitetura, com o RMSE de cada uma). Dica: use `summary_table(...)` em cada
dicionário de resultados e concatene os `DataFrame`s, adicionando uma coluna `"Dataset"`.


In [ ]:
# TODO: construa um único DataFrame comparando os 3 datasets x 3 arquiteturas (RMSE)
final_comparison = None
final_comparison


**Pergunta final** — Olhando a tabela consolidada: em qual(is) dataset(s) a diferença entre RNN Vanilla, LSTM e GRU foi maior, e em qual(is) foi menor? O que isso sugere sobre quando vale a pena pagar o custo extra de uma LSTM/GRU em vez de uma RNN simples?

---

**Sua resposta:**

_(escreva aqui)_

## 6. Encerramento

Você implementou e comparou RNN Vanilla, LSTM e GRU em três domínios diferentes — finanças, saúde pública e
clima — usando sempre o mesmo pipeline (`create_dataset` → tensor 3D → `DataLoader` → modelo → `train_model`
→ `evaluate_model`). A arquitetura muda; o pipeline ao redor dela, não.
